# Pakistani Politician Image Classifier – Complete Training Pipeline

An end-to-end facial recognition pipeline that classifies 16 Pakistani politicians
from face images using deep metric learning (ArcFace) and standard transfer learning.

All pipeline logic—data preparation, model definitions, training loops, and
evaluation—resides in the `training/` Python package. This notebook is a **thin
orchestration wrapper** that installs dependencies, imports the package, and
delegates execution to `main()`. The package can also be run standalone:

```bash
python training/main.py
```

| Model | Pretrain weights | Loss | Test Accuracy |
|---|---|---|---|
| InceptionResNetV1 | VGGFace2 | ArcFace | **97.06 %** |
| InceptionResNetV1 | CASIA-WebFace | ArcFace | **96.32 %** |
| ResNet-50 | ImageNet | CrossEntropy | **96.32 %** |

## Setup and Dependencies

The cells below install packages absent from the default Kaggle runtime:

- **`facenet-pytorch`** – provides the MTCNN face detector and the
  InceptionResNetV1 backbone pre-trained on VGGFace2 and CASIA-WebFace.
- **`imagehash`** – used for perceptual-hash (pHash) near-duplicate detection.

The remaining required packages (`timm`, `albumentations`) are installed
automatically by `main()` at runtime.

In [1]:
!pip install --no-deps facenet-pytorch imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.6 MB/s eta 0:00:0000:010:01


In [30]:
%pip install --upgrade torch torchvision --no-deps  # or simply: !pip install torch torchvision --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 97.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128
  Attempting uninstall: torch
    Found existing installation: torch 2.10.0+cu128
    Uninstalling torch-2.10.0+cu128:
      Successfully uninstalled torch-2.10.0+cu128
Note: you may need to restart the kernel to use updated packages.


## Modular Pipeline Wrapper

The cell below is the **sole execution entry point**. It:

1. Injects the correct directory into `sys.path` so that `import training`
   resolves on both Kaggle (package mounted as a dataset input or present in
   `/kaggle/working`) and local environments.
2. Imports `config` to confirm the package is reachable.
3. Calls `main()`, which executes the complete eight-stage pipeline
   (see *Pipeline Stages* below for details).

All outputs are written to `/kaggle/working/` (Kaggle) or
`project_outputs/` (local).

In [ ]:
# ============================================================
# MODULAR PIPELINE WRAPPER
# All training logic lives in the `training/` package.
# This notebook is a thin orchestration shim.
# ============================================================

import sys, os

# Inject package path for Kaggle environments
for _candidate in [
    '/kaggle/input',
    '/kaggle/working',
    os.path.dirname(os.path.dirname(os.path.abspath('__file__'))),
]:
    if os.path.isdir(os.path.join(_candidate, 'training')) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from training.config import config  # noqa – confirms package is importable
from training.main   import main

main()


---

## Pipeline Stages

The following sections document each stage executed automatically by `main()`.
No additional code is needed; this section exists for reference and reproducibility.

### Stage 1 – Data Loading

Raw images are copied from the Kaggle input directories into local `data/raw`
and `data/raw2` folders. The helper function `load_kaggle_data()` in
`training/data_prep.py` detects the Kaggle environment via
`KAGGLE_KERNEL_RUN_TYPE` and adjusts source paths accordingly.

### Stage 2 – Data Merging

Both raw image sources are merged into a single `data/raw_merged/` directory
using `merge_raw_folders()`. Images from the second source are renamed
sequentially (continuing from the last index in the first source) to avoid
filename collisions across the two datasets.

### Stage 3 – Face Alignment and Deduplication

**MTCNN alignment** (`align_faces_mtcnn()`):
- Detects faces using the MTCNN three-stage cascade detector.
- Discards images where zero or more than one face is detected.
- Aligns the detected face using five facial landmarks.
- Crops and resizes to **336 × 336 px** with a 0.2 margin ratio.
- Minimum image resolution gate: 60 × 60 px before alignment.

**pHash deduplication** (`deduplicate_phash()`):
- Computes a 64-bit perceptual hash for each aligned image.
- Removes near-duplicate pairs with a hash distance ≤ 5 to prevent
  data leakage across train/val/test splits.

### Stage 4 – Dataset Splitting

`split_dataset()` applies a stratified split over the aligned images:

| Split | Fraction |
|---|---|
| Train | 75 % |
| Validation | 15 % |
| Test | 10 % |

Classes with fewer than `MIN_IMAGES_FOR_SPLIT` (default: 5) images are
excluded. The directory structure written is `dataset/{train,val,test}/<class>/`.

### Stage 5 – Offline Data Augmentation

`run_offline_augmentation()` targets under-represented training classes.
For each class with fewer than 120 training images, it generates
**3 augmented copies** of every original image using an Albumentations
pipeline that includes:

- Geometric transforms: ShiftScaleRotate, HorizontalFlip, Perspective
- Colour jitter: RandomBrightnessContrast, HueSaturationValue, CLAHE
- Blur / noise: GaussianBlur, GaussNoise, ImageCompression

Online augmentation (random crops, flips, colour jitter) is additionally
applied to the training DataLoader at runtime.

### Stage 6 – Model Architecture

Three backbone families are supported:

**InceptionResNetV1 (ArcFace path)**
- Pre-trained weights: VGGFace2 or CASIA-WebFace (via `facenet-pytorch`).
- Wrapped by `FaceEmbeddingModel` which exposes a 512-dimensional embedding
  head and an optional classification head.
- Trained with ArcFace metric loss (see next section).

**EfficientNet-B3 (ArcFace path)**
- Loaded via `timm`; embedding size 512.
- Wrapped by `EfficientNetEmbeddingModel`.

**ResNet-50 / ResNet-152 / VGG-16 / ConvNeXt (CE path)**
- Loaded from `torchvision.models` or `timm` with ImageNet weights.
- Final fully-connected layer replaced with a 16-class head.
- Trained with cross-entropy loss and optional class weights.

### Loss Function – ArcFace

`ArcMarginProduct` adds an angular margin *m* = 0.3 to the ground-truth
class angle before applying the scale factor *s* = 64.0:

```
logit_i = s * cos(theta_i + m)   for the target class
logit_j = s * cos(theta_j)       for all other classes
```

The final loss is `F.cross_entropy(logits, labels, label_smoothing=0.1)`.
During validation and inference the margin is **not** applied—raw cosine
similarity logits are used instead, which avoids inflating validation loss.

### Training Strategy

| Setting | Value |
|---|---|
| Optimiser | AdamW |
| LR schedule | Cosine annealing |
| Head LR | 1 × 10⁻⁴ |
| Backbone unfreeze LR | 3 × 10⁻⁶ |
| Weight decay | 1 × 10⁻⁴ |
| Gradient clipping | max-norm = 1.0 |
| Mixed precision | `torch.cuda.amp` |
| MixUp | α = 0.2, probability = 0.5 |
| Early stopping | patience = 10 epochs |
| Epochs | 30 |

**Backbone freezing strategy:** layers are frozen for epochs 1–5. At epoch 6
the optimiser is rebuilt with discriminative learning rates: the classification
head uses LR = 1 × 10⁻⁴ while the backbone (ResNet layer4 / ArcFace backbone)
uses LR = 1 × 10⁻⁵ or 3 × 10⁻⁶.

### Evaluation and Metrics

`evaluate_model()` runs on the held-out test set after each model is trained:

- **Test-Time Augmentation (TTA):** 5-crop + horizontal flip (10 views per
  image); logits are averaged before taking the argmax.
- **Metrics:** overall accuracy, macro-averaged precision, recall, F1
  (via `sklearn.metrics`).
- **Confusion matrix:** saved as a seaborn heatmap to
  `{OUTPUT_DIR}/plots/{model}_confusion_matrix.png`.
- **Top-5 misclassified samples:** displayed inline with true and predicted labels.
- **Optional mislabeled audit** (`config.FLAG_MISLABELED = True`): scans the
  training set and flags images the model predicts with > 80 % confidence as a
  different class than their folder label.

### Training Loop Details

- **ArcFace validation:** cosine similarity logits (no margin) are used for
  validation accuracy, preventing artificially low validation scores.
- **Optimizer rebuild at epoch 6:** ArcFace class-weight parameters are
  explicitly included in the new optimizer parameter groups to prevent them
  being silently dropped during the backbone-unfreeze step.
- **Device consistency fix:** `class_weights` tensors are moved to the same
  device as the model before `nn.CrossEntropyLoss` is instantiated, resolving
  a runtime error present in the original monolithic notebook.
- **Best checkpoint:** the model state achieving the highest validation
  accuracy is restored at the end of training via `deepcopy`.

---

## Results and Export

After training completes, all outputs are available in `/kaggle/working/`:

| Directory | Contents |
|---|---|
| `models/` | Best model checkpoints (`{model}_best.pth`) |
| `plots/` | Training curves and confusion matrix heatmaps (PNG) |
| `results/` | `model_comparison.csv` with accuracy, precision, recall, F1 |

The two cells below create downloadable zip archives of these outputs directly
from the Kaggle notebook interface.

In [63]:
import shutil
import os

base_dir = "/kaggle/working"
zip_path = "/kaggle/working/project_outputs"

folders_to_zip = ["models", "plots", "results"]

# Create a temp directory to gather selected folders
temp_dir = "/kaggle/working/_temp_zip"
os.makedirs(temp_dir, exist_ok=True)

# Copy selected folders into temp dir
for folder in folders_to_zip:
    src = os.path.join(base_dir, folder)
    dst = os.path.join(temp_dir, folder)
    
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✓ Added: {folder}")
    else:
        print(f"Skipped (not found): {folder}")

# Create zip
shutil.make_archive(zip_path, 'zip', temp_dir)

print(f"\nZip created: {zip_path}.zip")

# Optional: remove temp folder
shutil.rmtree(temp_dir)

✓ Added: models
✓ Added: plots
✓ Added: results

Zip created: /kaggle/working/project_outputs.zip


In [64]:
import shutil, os
from IPython.display import FileLink

raw_merged_dir = "data/raw_merged"
zip_base = "/kaggle/working/raw_merged"          # without .zip extension
zip_path = zip_base + ".zip"

if os.path.isdir(raw_merged_dir):
    # Create the zip archive
    shutil.make_archive(zip_base, 'zip', raw_merged_dir)
    print(f"✔ Archive created: {zip_path}")
    
    # Show a clickable download link
    display(FileLink(zip_path))
else:
    print("⚠️ 'data/raw_merged' not found. Run the merge cell first.")

✔ Archive created: /kaggle/working/raw_merged.zip


/kaggle/working/raw_merged.zip

---

## Final Notes

This notebook is a **thin wrapper**. The underlying pipeline is a standalone,
fully modular Python package located in `training/`:

```
training/
  config.py      # Central hyperparameters and paths
  arcface.py     # ArcMarginProduct + ArcFaceLoss
  models.py      # Model factories and wrappers
  data_prep.py   # MTCNN alignment, pHash dedup, split, augmentation
  datasets.py    # PoliticianDataset and DataLoader factory
  training.py    # ArcFace and CrossEntropy training loops
  evaluate.py    # Evaluation, confusion matrix, mislabeled audit
  predict.py     # Single-image inference CLI
  main.py        # Orchestration entry point
```

To run outside Kaggle:

```bash
pip install -r requirements.txt
python training/main.py
```

Configuration (epochs, models, ArcFace margin, augmentation threshold, etc.)
is centralised in `training/config.py`.